In [0]:
dbutils.widgets.text("container", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("secret_scope", "")

_container = dbutils.widgets.get("container")
_storage_account = dbutils.widgets.get("storage_account")
_secret_scope = dbutils.widgets.get("secret_scope")

In [0]:
# ==============================================================================
# Production Auto Loader Pipeline with Schema Evolution
# ==============================================================================
import time

BASE_PATH = f"abfss://{_container}@{_storage_account}.dfs.core.windows.net"
LANDING_PATH = f"{BASE_PATH}/lab3/landing_zone/"
CHECKPOINT_EVO_PATH = f"{BASE_PATH}/lab3/_checkpoints/bronze_flights_evo/"
SCHEMA_EVO_LOCATION = f"{BASE_PATH}/lab3/_schemas/flights_stream_evo/"

def build_stream():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.schemaLocation", SCHEMA_EVO_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.allowOverwrites", "true")
        .load(LANDING_PATH)
    )

MAX_RETRIES = 5
for attempt in range(MAX_RETRIES):
    try:
        query_evo = (
            build_stream().writeStream.format("delta")
            .outputMode("append")
            .option("checkpointLocation", CHECKPOINT_EVO_PATH)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)
            .toTable("dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")
        )
        query_evo.awaitTermination()
        print("✅ Table successfully recreated and data ingested!")
        break
    except Exception as e:
        exc_text = str(e) + str(getattr(e, '__cause__', '') or '')
        if "UNKNOWN_FIELD_EXCEPTION" in exc_text or "NEW_FIELDS_IN_FILE" in exc_text:
            print(f"Schema evolution detected (attempt {attempt + 1}/{MAX_RETRIES}), retrying...")
            time.sleep(2)
        else:
            raise
else:
    raise RuntimeError("Max retries exceeded during schema evolution")

# 3. Print Streaming Statistics
progress_history = query_evo.recentProgress

if progress_history:
    total_rows = sum(p.get("numInputRows") or 0 for p in progress_history)
    batch_count = len(progress_history)
    
    # Grab execution timing from the active batches
    total_duration = sum(p.get("durationMs", {}).get("triggerExecution", 0) for p in progress_history)
    avg_rate = (total_rows / (total_duration / 1000.0)) if total_duration > 0 else 0.0

    print("=" * 60)
    print("      STREAMING BATCH METRICS REPORT")
    print("=" * 60)
    print(f"Total Micro-Batches    : {batch_count}")
    print(f"Total Rows Processed   : {total_rows}")
    print(f"Effective Processing   : {avg_rate:.2f} rows/sec")
    print(f"Total Execution Time   : {total_duration} ms")
    print("=" * 60)
else:
    print("ℹ️ No streaming progress history recorded.")

In [0]:
# dbutils.fs.rm(SCHEMA_EVO_LOCATION, recurse=True)
# dbutils.fs.rm(CHECKPOINT_EVO_PATH, recurse=True)
# spark.sql("DROP TABLE IF EXISTS dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")

In [0]:
%sql
-- 1. Check total row count (should be 10,030 if 3 files contained 10 rows each, or 10,003 if 1 row per file)
SELECT COUNT(*) FROM dbr_dev.joshuandegwa_bronze.lab3_flights_evolved;

-- 2. Inspect new columns and rescued data
-- SELECT icao24, callsign, aircraft_type, passenger_count, _rescued_data
-- FROM dbr_dev.joshuandegwa_bronze.lab3_flights_evolved
-- WHERE aircraft_type IS NOT NULL OR _rescued_data IS NOT NULL;

In [0]:
%sql
-- 1. Check updated table schema (Notice new columns 'aircraft_type', 'passenger_count', and '_rescued_data')
DESCRIBE joshuandegwa_bronze.lab3_flights_evolved;

In [0]:
%sql
SELECT COUNT(*) FROM joshuandegwa_bronze.lab3_flights_evolved;

In [0]:
# # Read the first few lines of batch 1001 directly to check headers
# batch_1001_path = f"{LANDING_PATH}schema_test_batch_1003.csv"
# lines = dbutils.fs.head(batch_1001_path, 500)
# print(lines)

In [0]:
# ==============================================================================
# EXPERIMENT CELL: Trigger Type Comparison
# ==============================================================================

# Choose mode to test: "availableNow" | "once" | "continuous"
TRIGGER_MODE = "availableNow" 

stream_reader = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaLocation", SCHEMA_EVO_LOCATION)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.allowOverwrites", "true")
    .load(LANDING_PATH)
)

writer = (
    stream_reader.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_EVO_PATH)
    .option("mergeSchema", "true")
)

# ------------------------------------------------------------------------------
# TRIGGER MODE 1: availableNow
# WHAT HAPPENS: Auto Loader scans the landing zone for all unconsumed files,
# processes them incrementally across multiple micro-batches until the backlog
# reaches 0, and then shuts down automatically. Best for cost-efficient batch jobs.
# ------------------------------------------------------------------------------
if TRIGGER_MODE == "availableNow":
    print("🚀 Running with trigger(availableNow=True)... [Drains all backlog files, then stops]")
    q_exp = writer.trigger(availableNow=True).toTable("dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")
    q_exp.awaitTermination()

# ------------------------------------------------------------------------------
# TRIGGER MODE 2: once
# WHAT HAPPENS: Auto Loader reads ONLY a single micro-batch of newly arrived files 
# and stops immediately, even if more files remain in the landing directory.
# (Legacy option; availableNow is preferred in modern Databricks).
# ------------------------------------------------------------------------------
elif TRIGGER_MODE == "once":
    print("🚀 Running with trigger(once=True)... [Processes 1 micro-batch only, then stops]")
    q_exp = writer.trigger(once=True).toTable("dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")
    q_exp.awaitTermination()

# ------------------------------------------------------------------------------
# TRIGGER MODE 3: processingTime (Continuous)
# WHAT HAPPENS: The stream stays active indefinitely in memory, waking up at 
# fixed intervals (e.g., every 10 seconds) to check for new files and ingest 
# them in near real-time. Must be manually stopped (query.stop()).
# ------------------------------------------------------------------------------
elif TRIGGER_MODE == "continuous":
    import time
    print("⚡ Starting Continuous Stream (10s interval)... [Runs continuously until stopped]")
    q_exp = writer.trigger(processingTime="10 seconds").toTable("dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")
    time.sleep(30)  # Runs continuous polling for 30 seconds for test purposes
    q_exp.stop()
    print("🛑 Continuous stream stopped.")

print(f"✅ Trigger experiment for '{TRIGGER_MODE}' complete!")

In [0]:
# Purge Notebook 2 & 3 tables
spark.sql("DROP TABLE IF EXISTS dbr_dev.joshuandegwa_bronze.lab3_flights_raw")
spark.sql("DROP TABLE IF EXISTS dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")